In [1]:
import os
from pathlib import Path

In [2]:
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from PIL import Image
from rasterio.features import rasterize
from tqdm import tqdm

In [3]:
CSV_PATH = "data/legend_class_geo.csv"
TIFF_PATH = "data/el_harrach_georef.tif"
GEOJSON_DIR = Path("output/vect/poly")
OUTPUT_DIR = Path("output/viz")
TARGET_CRS = "EPSG:3857"

In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
def save_png(arr: np.ndarray, path: Path):
    Image.fromarray(arr).save(path, format="PNG", optimize=False)

In [6]:
# -----------------------
# Load class → value map
# -----------------------
df = pd.read_csv(CSV_PATH).dropna(subset=["class", "z"])
z_map = dict(zip(df["class"].astype(str), df["z"].astype(int)))

In [7]:
# -----------------------
# Load raster reference
# -----------------------
with rasterio.open(TIFF_PATH) as src:
    transform = src.transform
    crs = src.crs
    h, w = src.height, src.width

    if src.count >= 3:
        rgb_base = np.transpose(src.read([1, 2, 3]), (1, 2, 0)).astype(np.uint8)
    else:
        band = src.read(1).astype(np.uint8)
        rgb_base = np.stack([band, band, band], axis=-1)

In [8]:
if str(crs) != TARGET_CRS:
    raise ValueError(f"Expected raster CRS {TARGET_CRS}, got {crs}")

In [9]:
# -----------------------
# Load ALL GeoJSON once
# -----------------------
geojson_map = {
    p.stem: gpd.read_file(p)
    for p in GEOJSON_DIR.glob("*.geojson")
}

In [10]:
# -----------------------
# Build ONE raster (key optimization)
# -----------------------
shapes = []

In [11]:
for cls_name, z in z_map.items():
    gdf = geojson_map.get(cls_name)
    if gdf is None:
        continue

    gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty]
    if gdf.empty:
        continue

    if gdf.crs is None:
        raise ValueError(f"{cls_name}.geojson has no CRS")
    if str(gdf.crs) != TARGET_CRS:
        raise ValueError(f"{cls_name}.geojson CRS must be {TARGET_CRS}, got {gdf.crs}")

    shapes.extend((geom, z) for geom in gdf.geometry)

In [12]:
print(f"Rasterizing {len(shapes)} geometries...")

Rasterizing 20581 geometries...


In [13]:
class_raster = rasterize(
    shapes,
    out_shape=(h, w),
    transform=transform,
    fill=0,
    dtype=np.uint16,   # important: supports many classes
)

In [14]:
# -----------------------
# Generate outputs per class (fast NumPy ops)
# -----------------------
for cls_name, z in tqdm(z_map.items()):

    mask = (class_raster == z)
    if not mask.any():
        continue

    # RGBA mask
    rgba = np.zeros((h, w, 4), dtype=np.uint8)
    rgba[mask] = [255, 0, 0, 255]

    # RGB overlay
    overlay = rgb_base.copy()
    overlay[mask] = [255, 0, 0]

    save_png(rgba, OUTPUT_DIR / f"{z}_{cls_name}_mask.png")
    save_png(overlay, OUTPUT_DIR / f"{z}_{cls_name}_overlay.png")

  0%|                                         | 0/11 [00:00<?, ?it/s]

  9%|███                              | 1/11 [00:06<01:09,  6.92s/it]

 18%|██████                           | 2/11 [00:11<00:48,  5.35s/it]

 27%|█████████                        | 3/11 [00:14<00:36,  4.56s/it]

 36%|████████████                     | 4/11 [00:19<00:31,  4.49s/it]

 45%|███████████████                  | 5/11 [00:22<00:24,  4.11s/it]

 55%|██████████████████               | 6/11 [00:26<00:20,  4.06s/it]

 64%|█████████████████████            | 7/11 [00:31<00:17,  4.32s/it]

100%|████████████████████████████████| 11/11 [00:31<00:00,  2.87s/it]